In [12]:
# ─── CELL 1: IMPORTS ──────────────────────────────────────────────────────────

import csv
import os
import json
import time
import random
import pandas as pd
import sys
from datetime import datetime


print("✅ Imports done")

# ─── CELL 2: LOAD VAL_DF ──────────────────────────────────────────────────────

VAL_DF_PATH = "val_df.csv"  # ← update path
val_df = pd.read_csv(VAL_DF_PATH)
print(f"✅ val_df loaded: {len(val_df)} rows")
print(f"   Glucose range: {val_df['glucose'].min():.1f} – {val_df['glucose'].max():.1f} mg/dL")



✅ Imports done
✅ val_df loaded: 9047 rows
   Glucose range: 39.6 – 473.4 mg/dL


In [22]:
# ─── CELL 3: IMPORT AGENTS FROM YOUR MAIN NOTEBOOK ───────────────────────────
# Option A — if agents are defined in a .py file:
# Remove cached version if it exists
if 'capstone_agents_pipeline' in sys.modules:
    del sys.modules['capstone_agents_pipeline']
from capstone_agents_pipeline import run_main_with_safety, token_counter


✅ Gemini API key setup complete from Secret Manager.
✅ Food API key setup complete from Secret Manager.
✅ ADK components imported successfully.
✅ Retry config for Agents created successfully.
✅ AlertAgent created.
✅ InsulinAgent created.
✅ search_food_by_carbs() created successfully
✅ MealAgent created.
✅ ExerciseAgent created.
✅ SafetyAgent created.
✅ Validation data loaded: 9047 rows, 53 features
✅ Prediction model loaded
✅ predict_glucose defined — row_number in, current + 4 future points out (mg/dL)
✅ Main_agent created.
✅ Formatter Agent created.


/opt/conda/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator LinearRegression from version 1.7.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [23]:
# ─── CELL 4: ROW POOL SETUP ───────────────────────────────────────────────────

def find_rows(min_mgdl: float, max_mgdl: float, n: int = 20) -> list:
    """Both current glucose AND prediction must fall in [min_mgdl, max_mgdl]."""
    mask = (
        (val_df['glucose']    >= min_mgdl) & (val_df['glucose']    <= max_mgdl) &
        (val_df['prediction'] >= min_mgdl) & (val_df['prediction'] <= max_mgdl)
    )
    rows = val_df[mask].index.tolist()
    random.shuffle(rows)
    return rows[:n]

ROWS_NORMAL = find_rows(90,  150)
ROWS_HYPO   = find_rows(39,   80)
ROWS_HYPER  = find_rows(180, 400)

def safe_row(pool, idx, fallback_pool=None):
    if pool and idx < len(pool):
        return pool[idx]
    if fallback_pool:
        return fallback_pool[0]
    return ROWS_NORMAL[0]

print(f"   Normal  (90–150): {len(ROWS_NORMAL)} rows")
print(f"   Hypo    (39–80):  {len(ROWS_HYPO)} rows")
print(f"   Hyper   (180–400):{len(ROWS_HYPER)} rows")

# ── Day constants ─────────────────────────────────────────────────────────────
MON = "Monday"
TUE = "Tuesday"
WED = "Wednesday"
THU = "Thursday"
FRI = "Friday"
SAT = "Saturday"
SUN = "Sunday"

# ── Long-acting insulin templates ─────────────────────────────────────────────
LAI_NIGHTLY_9PM = "Yes, every night 9PM ET"
LAI_MORNING_7AM = "Yes, every morning 7AM ET"
LAI_NO          = "no"

# ── GLP-1 templates ───────────────────────────────────────────────────────────
GLP1_DAILY      = "daily"
GLP1_WEEKLY_SAT = "weekly on Saturdays"
GLP1_WEEKLY_MON = "weekly on Mondays"
GLP1_WEEKLY_FRI = "weekly on Fridays"
GLP1_PREMEAL    = "pre-meal"
GLP1_NO         = "no"

# ─── CELL 5: TEST CASE BUILDER ────────────────────────────────────────────────

def build_input(
    label,
    last_meal,
    current_time,           # format: "Saturday, 6:30 PM ET"
    row_number,
    diet                = "Non-Veg",
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM,
    glp1                = GLP1_NO,
    weight              = 75,
    height              = 1.65
):
    return {
        "label":      label,
        "row_number": row_number,
        "input": f"""
last_meal = {last_meal}
current_time = {current_time}
row_number = {row_number}
weight = {weight} kg
height = {height} m
diet = {diet}
usual_meal_times:
  breakfast = 7:00 AM ET
  lunch = 12:30 PM ET
  dinner = 7:00 PM ET
oral_medication = {oral_medication}
insulin = {insulin}
long_acting_insulin = {long_acting_insulin}
glp1 = {glp1}
"""
    }


   Normal  (90–150): 20 rows
   Hypo    (39–80):  20 rows
   Hyper   (180–400):20 rows


In [24]:
test_cases = []

# ══════════════════════════════════════════════════════════════════════════════
# GROUP 1: NORMAL GLUCOSE — 10 cases
# Expected: carb_rule=NORMAL, balanced meal, no insulin correction
# ══════════════════════════════════════════════════════════════════════════════

test_cases.append(build_input(
    label               = "NORM-BRK-01 | Normal→Normal | Monday 30min BEFORE breakfast | oral+insulin+LAI",
    last_meal           = "Dinner at 7:00 PM ET previous day",
    current_time        = f"{MON}, 6:30 AM ET",
    row_number          = safe_row(ROWS_NORMAL, 0),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))

test_cases.append(build_input(
    label               = "NORM-BRK-02 | Normal→Normal | Wednesday AT breakfast | no medication",
    last_meal           = "Dinner at 7:00 PM ET previous day",
    current_time        = f"{WED}, 7:00 AM ET",
    row_number          = safe_row(ROWS_NORMAL, 1),
    oral_medication     = "none",
    insulin             = "no",
    long_acting_insulin = LAI_NO
))

test_cases.append(build_input(
    label               = "NORM-LCH-01 | Normal→Normal | Tuesday 30min BEFORE lunch | oral+insulin+LAI",
    last_meal           = "Breakfast at 7:00 AM ET",
    current_time        = f"{TUE}, 12:00 PM ET",
    row_number          = safe_row(ROWS_NORMAL, 2),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))

test_cases.append(build_input(
    label               = "NORM-LCH-02 | Normal→Normal | Monday PAST lunch not taken | oral+insulin+LAI",
    last_meal           = "Breakfast at 7:00 AM ET",
    current_time        = f"{MON}, 1:30 PM ET",
    row_number          = safe_row(ROWS_NORMAL, 3),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))

test_cases.append(build_input(
    label               = "NORM-DIN-01 | Normal→Normal | Friday 30min BEFORE dinner | oral+insulin+LAI",
    last_meal           = "Lunch at 12:30 PM ET",
    current_time        = f"{FRI}, 6:30 PM ET",
    row_number          = safe_row(ROWS_NORMAL, 4),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))

test_cases.append(build_input(
    label               = "NORM-DIN-02 | Normal→Normal | Sunday AT dinner | no medication",
    last_meal           = "Lunch at 12:30 PM ET",
    current_time        = f"{SUN}, 7:00 PM ET",
    row_number          = safe_row(ROWS_NORMAL, 5),
    oral_medication     = "none",
    insulin             = "no",
    long_acting_insulin = LAI_NO
))

test_cases.append(build_input(
    label               = "NORM-VEG-01 | Normal→Normal | Tuesday BEFORE lunch | Veg diet | oral+insulin",
    last_meal           = "Breakfast at 7:00 AM ET",
    current_time        = f"{TUE}, 12:00 PM ET",
    row_number          = safe_row(ROWS_NORMAL, 6),
    diet                = "Veg",
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))

test_cases.append(build_input(
    label               = "NORM-VGN-01 | Normal→Normal | Wednesday BEFORE lunch | Vegan diet | oral+insulin",
    last_meal           = "Breakfast at 7:00 AM ET",
    current_time        = f"{WED}, 12:00 PM ET",
    row_number          = safe_row(ROWS_NORMAL, 7),
    diet                = "Vegan",
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))

test_cases.append(build_input(
    label               = "NORM-INS-01 | Normal→Normal | Thursday BEFORE lunch | insulin ONLY no oral",
    last_meal           = "Breakfast at 7:00 AM ET",
    current_time        = f"{THU}, 12:00 PM ET",
    row_number          = safe_row(ROWS_NORMAL, 8),
    oral_medication     = "none",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))

test_cases.append(build_input(
    label               = "NORM-ORL-01 | Normal→Normal | Friday BEFORE dinner | oral ONLY no short-acting",
    last_meal           = "Lunch at 12:30 PM ET",
    current_time        = f"{FRI}, 6:30 PM ET",
    row_number          = safe_row(ROWS_NORMAL, 9),
    oral_medication     = "pre-meal",
    insulin             = "no",
    long_acting_insulin = LAI_NIGHTLY_9PM
))

# ══════════════════════════════════════════════════════════════════════════════
# GROUP 2: HYPOGLYCEMIA — 12 cases
# Expected: carb_rule=LOW, fast-acting carbs, insulin=0, exercise=unsafe
# ══════════════════════════════════════════════════════════════════════════════

test_cases.append(build_input(
    label               = "HYPO-BRK-01 | Hypo→Hypo | Monday 30min BEFORE breakfast | oral+insulin+LAI",
    last_meal           = "Dinner at 7:00 PM ET previous day",
    current_time        = f"{MON}, 6:30 AM ET",
    row_number          = safe_row(ROWS_HYPO, 0),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))

test_cases.append(build_input(
    label               = "HYPO-LCH-01 | Hypo→Hypo | Friday 30min BEFORE lunch | oral+insulin+LAI",
    last_meal           = "Breakfast at 7:00 AM ET",
    current_time        = f"{FRI}, 12:00 PM ET",
    row_number          = safe_row(ROWS_HYPO, 1),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))

test_cases.append(build_input(
    label               = "HYPO-LCH-02 | Hypo→Hypo | Thursday PAST lunch not taken | oral+insulin+LAI",
    last_meal           = "Breakfast at 7:00 AM ET",
    current_time        = f"{THU}, 1:30 PM ET",
    row_number          = safe_row(ROWS_HYPO, 2),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))

test_cases.append(build_input(
    label               = "HYPO-DIN-01 | Hypo→Hypo | Thursday 30min BEFORE dinner | oral+insulin+LAI",
    last_meal           = "Lunch at 12:30 PM ET",
    current_time        = f"{THU}, 6:30 PM ET",
    row_number          = safe_row(ROWS_HYPO, 3),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))

test_cases.append(build_input(
    label               = "HYPO-LAI-01 | Hypo→Hypo | Saturday 30min BEFORE 9PM LAI | critical — hypo + LAI due",
    last_meal           = "Dinner at 7:00 PM ET",
    current_time        = f"{SAT}, 8:30 PM ET",
    row_number          = safe_row(ROWS_HYPO, 4),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))

test_cases.append(build_input(
    label               = "HYPO-LAI-02 | Hypo→Hypo | Monday 1hr AFTER 9PM LAI | hypo post-LAI high risk",
    last_meal           = "Dinner at 7:00 PM ET",
    current_time        = f"{MON}, 10:00 PM ET",
    row_number          = safe_row(ROWS_HYPO, 5),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))

test_cases.append(build_input(
    label               = "HYPO-GATE-01 | Hypo→Hypo | Friday 3hrs BEFORE lunch | hypo overrides timing gate — meal required",
    last_meal           = "Breakfast at 7:00 AM ET",
    current_time        = f"{FRI}, 9:30 AM ET",
    row_number          = safe_row(ROWS_HYPO, 6),
    oral_medication     = "none",
    insulin             = "no",
    long_acting_insulin = LAI_NO
))

test_cases.append(build_input(
    label               = "HYPO-GATE-02 | Hypo→Hypo | Thursday recently ate 30min ago | hypo overrides — still needs fast carbs",
    last_meal           = "Lunch at 12:00 PM ET",
    current_time        = f"{THU}, 12:30 PM ET",
    row_number          = safe_row(ROWS_HYPO, 7),
    oral_medication     = "none",
    insulin             = "no",
    long_acting_insulin = LAI_NO
))

test_cases.append(build_input(
    label               = "HYPO-VEG-01 | Hypo→Hypo | Monday BEFORE lunch | Veg diet | fast carbs must be veg-compliant",
    last_meal           = "Breakfast at 7:00 AM ET",
    current_time        = f"{MON}, 12:00 PM ET",
    row_number          = safe_row(ROWS_HYPO, 8),
    diet                = "Veg",
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))

test_cases.append(build_input(
    label               = "HYPO-VGN-01 | Hypo→Hypo | Tuesday BEFORE dinner | Vegan diet | fast carbs must be vegan-compliant",
    last_meal           = "Lunch at 12:30 PM ET",
    current_time        = f"{TUE}, 6:30 PM ET",
    row_number          = safe_row(ROWS_HYPO, 9),
    diet                = "Vegan",
    oral_medication     = "none",
    insulin             = "no",
    long_acting_insulin = LAI_NO
))

test_cases.append(build_input(
    label               = "HYPO-GLP1-01 | Hypo→Hypo | Saturday BEFORE lunch | daily GLP-1 | hypo overrides insulin",
    last_meal           = "Breakfast at 7:00 AM ET",
    current_time        = f"{SAT}, 12:00 PM ET",
    row_number          = safe_row(ROWS_HYPO, 10),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM,
    glp1                = GLP1_DAILY
))

test_cases.append(build_input(
    label               = "HYPO-GLP1-02 | Hypo→Hypo | Saturday BEFORE dinner | weekly GLP-1 DUE TODAY ✅ | hypo overrides insulin",
    last_meal           = "Lunch at 12:30 PM ET",
    current_time        = f"{SAT}, 6:30 PM ET",
    row_number          = safe_row(ROWS_HYPO, 11),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM,
    glp1                = GLP1_WEEKLY_SAT
))

# ══════════════════════════════════════════════════════════════════════════════
# GROUP 3: HYPERGLYCEMIA — 13 cases
# Expected: carb_rule=HIGH, protein+veg only, insulin correction required
# ══════════════════════════════════════════════════════════════════════════════

test_cases.append(build_input(
    label               = "HYPR-BRK-01 | Hyper→Hyper | Wednesday 30min BEFORE breakfast | oral+insulin+LAI",
    last_meal           = "Dinner at 7:00 PM ET previous day",
    current_time        = f"{WED}, 6:30 AM ET",
    row_number          = safe_row(ROWS_HYPER, 0),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))

test_cases.append(build_input(
    label               = "HYPR-LCH-01 | Hyper→Hyper | Monday 30min BEFORE lunch | oral+insulin+LAI",
    last_meal           = "Breakfast at 7:00 AM ET",
    current_time        = f"{MON}, 12:00 PM ET",
    row_number          = safe_row(ROWS_HYPER, 1),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))

test_cases.append(build_input(
    label               = "HYPR-LCH-02 | Hyper→Hyper | Tuesday PAST lunch not taken | oral+insulin+LAI",
    last_meal           = "Breakfast at 7:00 AM ET",
    current_time        = f"{TUE}, 1:30 PM ET",
    row_number          = safe_row(ROWS_HYPER, 2),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))

test_cases.append(build_input(
    label               = "HYPR-DIN-01 | Hyper→Hyper | Tuesday 30min BEFORE dinner | oral+insulin+LAI",
    last_meal           = "Lunch at 12:30 PM ET",
    current_time        = f"{TUE}, 6:30 PM ET",
    row_number          = safe_row(ROWS_HYPER, 3),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))

test_cases.append(build_input(
    label               = "HYPR-WIN-01 | Hyper→Hyper | Friday 3hrs BEFORE lunch | outside window — no meal or insulin",
    last_meal           = "Breakfast at 7:00 AM ET",
    current_time        = f"{FRI}, 9:30 AM ET",
    row_number          = safe_row(ROWS_HYPER, 4),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))

test_cases.append(build_input(
    label               = "HYPR-WIN-02 | Hyper→Hyper | Monday late night 11PM | outside all meal windows",
    last_meal           = "Dinner at 7:00 PM ET",
    current_time        = f"{MON}, 11:00 PM ET",
    row_number          = safe_row(ROWS_HYPER, 5),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))

test_cases.append(build_input(
    label               = "HYPR-LAI-01 | Hyper→Hyper | Thursday 30min BEFORE 9PM LAI | hyper + LAI due",
    last_meal           = "Dinner at 7:00 PM ET",
    current_time        = f"{THU}, 8:30 PM ET",
    row_number          = safe_row(ROWS_HYPER, 6),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))

test_cases.append(build_input(
    label               = "HYPR-VEG-01 | Hyper→Hyper | Monday BEFORE lunch | Veg diet | protein+veg only",
    last_meal           = "Breakfast at 7:00 AM ET",
    current_time        = f"{MON}, 12:00 PM ET",
    row_number          = safe_row(ROWS_HYPER, 7),
    diet                = "Veg",
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))

test_cases.append(build_input(
    label               = "HYPR-VGN-01 | Hyper→Hyper | Wednesday BEFORE dinner | Vegan diet | protein+veg only",
    last_meal           = "Lunch at 12:30 PM ET",
    current_time        = f"{WED}, 6:30 PM ET",
    row_number          = safe_row(ROWS_HYPER, 8),
    diet                = "Vegan",
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))

test_cases.append(build_input(
    label               = "HYPR-ORL-01 | Hyper→Hyper | Tuesday BEFORE lunch | oral ONLY no short-acting insulin",
    last_meal           = "Breakfast at 7:00 AM ET",
    current_time        = f"{TUE}, 12:00 PM ET",
    row_number          = safe_row(ROWS_HYPER, 9),
    oral_medication     = "pre-meal",
    insulin             = "no",
    long_acting_insulin = LAI_NIGHTLY_9PM
))

test_cases.append(build_input(
    label               = "HYPR-INS-01 | Hyper→Hyper | Friday BEFORE dinner | insulin ONLY no oral",
    last_meal           = "Lunch at 12:30 PM ET",
    current_time        = f"{FRI}, 6:30 PM ET",
    row_number          = safe_row(ROWS_HYPER, 10),
    oral_medication     = "none",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))

test_cases.append(build_input(
    label               = "HYPR-GLP1-01 | Hyper→Hyper | Thursday BEFORE lunch | daily GLP-1 + oral + insulin",
    last_meal           = "Breakfast at 7:00 AM ET",
    current_time        = f"{THU}, 12:00 PM ET",
    row_number          = safe_row(ROWS_HYPER, 11),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM,
    glp1                = GLP1_DAILY
))

test_cases.append(build_input(
    label               = "HYPR-GLP1-02 | Hyper→Hyper | Saturday BEFORE dinner | weekly GLP-1 DUE TODAY ✅ + oral + insulin",
    last_meal           = "Lunch at 12:30 PM ET",
    current_time        = f"{SAT}, 6:30 PM ET",
    row_number          = safe_row(ROWS_HYPER, 12),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM,
    glp1                = GLP1_WEEKLY_SAT
))

# ══════════════════════════════════════════════════════════════════════════════
# GROUP 4: LONG-ACTING INSULIN — 4 cases
# ══════════════════════════════════════════════════════════════════════════════

test_cases.append(build_input(
    label               = "LAI-01 | Normal→Normal | Wednesday 30min BEFORE 9PM LAI | all medications",
    last_meal           = "Dinner at 7:00 PM ET",
    current_time        = f"{WED}, 8:30 PM ET",
    row_number          = safe_row(ROWS_NORMAL, 10),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))

test_cases.append(build_input(
    label               = "LAI-02 | Normal→Normal | Tuesday AT 9PM LAI | long-acting ONLY no short-acting",
    last_meal           = "Dinner at 7:00 PM ET",
    current_time        = f"{TUE}, 9:00 PM ET",
    row_number          = safe_row(ROWS_NORMAL, 11),
    oral_medication     = "none",
    insulin             = "no",
    long_acting_insulin = LAI_NIGHTLY_9PM
))

test_cases.append(build_input(
    label               = "LAI-03 | Normal→Normal | Friday 2hrs BEFORE 9PM LAI | outside window — no LAI alert",
    last_meal           = "Dinner at 7:00 PM ET",
    current_time        = f"{FRI}, 7:00 PM ET",
    row_number          = safe_row(ROWS_NORMAL, 12),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))

test_cases.append(build_input(
    label               = "LAI-04 | Normal→Normal | Wednesday AT 7AM LAI | morning long-acting dose",
    last_meal           = "Dinner at 7:00 PM ET previous day",
    current_time        = f"{WED}, 7:00 AM ET",
    row_number          = safe_row(ROWS_NORMAL, 13),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_MORNING_7AM
))

# ══════════════════════════════════════════════════════════════════════════════
# GROUP 5: GLP-1 — 3 cases
# ══════════════════════════════════════════════════════════════════════════════

test_cases.append(build_input(
    label               = "GLP1-01 | Normal→Normal | Monday BEFORE lunch | daily GLP-1 + oral + insulin",
    last_meal           = "Breakfast at 7:00 AM ET",
    current_time        = f"{MON}, 12:00 PM ET",
    row_number          = safe_row(ROWS_NORMAL, 14),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM,
    glp1                = GLP1_DAILY
))

test_cases.append(build_input(
    label               = "GLP1-02 | Normal→Normal | Saturday BEFORE lunch | weekly GLP-1 DUE TODAY ✅",
    last_meal           = "Breakfast at 7:00 AM ET",
    current_time        = f"{SAT}, 12:00 PM ET",
    row_number          = safe_row(ROWS_NORMAL, 15),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM,
    glp1                = GLP1_WEEKLY_SAT
))

test_cases.append(build_input(
    label               = "GLP1-03 | Normal→Normal | Tuesday BEFORE lunch | weekly GLP-1 NOT TODAY ❌ (Sat scheduled)",
    last_meal           = "Breakfast at 7:00 AM ET",
    current_time        = f"{TUE}, 12:00 PM ET",
    row_number          = safe_row(ROWS_NORMAL, 16),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM,
    glp1                = GLP1_WEEKLY_SAT
))

# ══════════════════════════════════════════════════════════════════════════════
# GROUP 6: EDGE CASES — 8 cases
# ══════════════════════════════════════════════════════════════════════════════

test_cases.append(build_input(
    label               = "EDGE-01 | Normal→Normal | Tuesday recently ate 30min ago | no meal or insulin expected",
    last_meal           = "Lunch at 12:00 PM ET",
    current_time        = f"{TUE}, 12:30 PM ET",
    row_number          = safe_row(ROWS_NORMAL, 17),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))

test_cases.append(build_input(
    label               = "EDGE-02 | Normal→Normal | Friday 3hrs BEFORE lunch | no meal or insulin expected",
    last_meal           = "Breakfast at 7:00 AM ET",
    current_time        = f"{FRI}, 9:30 AM ET",
    row_number          = safe_row(ROWS_NORMAL, 18),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))

test_cases.append(build_input(
    label               = "EDGE-03 | Hypo→Hypo | Wednesday 3hrs BEFORE lunch | hypo overrides timing gate — meal required",
    last_meal           = "Breakfast at 7:00 AM ET",
    current_time        = f"{WED}, 9:30 AM ET",
    row_number          = safe_row(ROWS_HYPO, 12),
    oral_medication     = "none",
    insulin             = "no",
    long_acting_insulin = LAI_NO
))

test_cases.append(build_input(
    label               = "EDGE-04 | Normal→Normal | Thursday 60min BEFORE 9PM LAI | exactly at window — LAI alert expected",
    last_meal           = "Dinner at 7:00 PM ET",
    current_time        = f"{THU}, 8:00 PM ET",
    row_number          = safe_row(ROWS_NORMAL, 19),
    oral_medication     = "none",
    insulin             = "no",
    long_acting_insulin = LAI_NIGHTLY_9PM
))

test_cases.append(build_input(
    label               = "EDGE-05 | Normal→Normal | Friday 61min BEFORE 9PM LAI | just outside window — no LAI alert",
    last_meal           = "Dinner at 7:00 PM ET",
    current_time        = f"{FRI}, 7:59 PM ET",
    row_number          = safe_row(ROWS_NORMAL, 0),
    oral_medication     = "none",
    insulin             = "no",
    long_acting_insulin = LAI_NIGHTLY_9PM
))

test_cases.append(build_input(
    label               = "EDGE-06 | Hypo→Hypo | Wednesday AT 9PM LAI | hypo + LAI due simultaneously",
    last_meal           = "Dinner at 7:00 PM ET",
    current_time        = f"{WED}, 9:00 PM ET",
    row_number          = safe_row(ROWS_HYPO, 13),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM
))

test_cases.append(build_input(
    label               = "EDGE-07 | Normal→Normal | Sunday BEFORE lunch | no medications at all — no alert expected",
    last_meal           = "Breakfast at 7:00 AM ET",
    current_time        = f"{SUN}, 12:00 PM ET",
    row_number          = safe_row(ROWS_NORMAL, 1),
    oral_medication     = "none",
    insulin             = "no",
    long_acting_insulin = LAI_NO,
    glp1                = GLP1_NO
))

test_cases.append(build_input(
    label               = "EDGE-08 | Normal→Normal | Wednesday BEFORE dinner | weekly GLP-1 NOT TODAY ❌ — GLP-1 must be silent",
    last_meal           = "Lunch at 12:30 PM ET",
    current_time        = f"{WED}, 6:30 PM ET",
    row_number          = safe_row(ROWS_NORMAL, 2),
    oral_medication     = "pre-meal",
    insulin             = "yes",
    long_acting_insulin = LAI_NIGHTLY_9PM,
    glp1                = GLP1_WEEKLY_SAT
))

# ─── SUMMARY ──────────────────────────────────────────────────────────────────

print(f"\n{'='*70}")
print(f"TOTAL TEST CASES: {len(test_cases)}")
print(f"{'='*70}")

groups = {
    "GROUP 1 — Normal glucose":      [tc for tc in test_cases if tc['label'].startswith('NORM')],
    "GROUP 2 — Hypoglycemia":        [tc for tc in test_cases if tc['label'].startswith('HYPO')],
    "GROUP 3 — Hyperglycemia":       [tc for tc in test_cases if tc['label'].startswith('HYPR')],
    "GROUP 4 — Long-acting insulin": [tc for tc in test_cases if tc['label'].startswith('LAI')],
    "GROUP 5 — GLP-1":               [tc for tc in test_cases if tc['label'].startswith('GLP1')],
    "GROUP 6 — Edge cases":          [tc for tc in test_cases if tc['label'].startswith('EDGE')],
}

for name, cases in groups.items():
    print(f"  {name}: {len(cases)} cases")

print(f"\n── Coverage mix ─────────────────────────────────────────────────────")
print(f"  Meal slots (brk/lch/din):     all 3 covered in every glucose group")
print(f"  Glucose categories:           Normal / Hypo / Hyper")
print(f"  Diet variations:              Non-Veg / Veg / Vegan")
print(f"  Medication configs:           oral+insulin / oral only / insulin only / none")
print(f"  LAI timing:                   30min before / AT / 60min boundary / outside")
print(f"  GLP-1:                        daily / weekly due / weekly NOT due")
print(f"  Timing gate:                  recently ate / 3hrs early / hypo override")
print(f"  Simultaneous conditions:      hypo+LAI / hyper+LAI")

print(f"\n── Row pool check ───────────────────────────────────────────────────")
for pool_name, pool, needed in [
    ("ROWS_NORMAL", ROWS_NORMAL, 20),
    ("ROWS_HYPO",   ROWS_HYPO,   14),
    ("ROWS_HYPER",  ROWS_HYPER,  13),
]:
    status = "✅" if len(pool) >= needed else "⚠️ "
    print(f"  {status} {pool_name}: {len(pool)} available, {needed} needed")



TOTAL TEST CASES: 50
  GROUP 1 — Normal glucose: 10 cases
  GROUP 2 — Hypoglycemia: 12 cases
  GROUP 3 — Hyperglycemia: 13 cases
  GROUP 4 — Long-acting insulin: 4 cases
  GROUP 5 — GLP-1: 3 cases
  GROUP 6 — Edge cases: 8 cases

── Coverage mix ─────────────────────────────────────────────────────
  Meal slots (brk/lch/din):     all 3 covered in every glucose group
  Glucose categories:           Normal / Hypo / Hyper
  Diet variations:              Non-Veg / Veg / Vegan
  Medication configs:           oral+insulin / oral only / insulin only / none
  LAI timing:                   30min before / AT / 60min boundary / outside
  GLP-1:                        daily / weekly due / weekly NOT due
  Timing gate:                  recently ate / 3hrs early / hypo override
  Simultaneous conditions:      hypo+LAI / hyper+LAI

── Row pool check ───────────────────────────────────────────────────
  ✅ ROWS_NORMAL: 20 available, 20 needed
  ✅ ROWS_HYPO: 20 available, 14 needed
  ✅ ROWS_HYPER: 20 a

In [25]:
# ─── CELL 8: CSV HELPERS ──────────────────────────────────────────────────────

TEST_CSV_FILE = "test_results.csv"
TEST_CSV_HEADERS = [
    "run_timestamp",
    "test_id",
    "label",
    "row_number",
    "current_glucose",
    "glucose_at_meal_time",
    "glucose_category",
    "carb_rule",
    "meal_timing",
    "oral_medication",
    "insulin",
    "long_acting_insulin",
    "glp1",
    "diet",
    "elapsed_seconds",
    "input_tokens",          
    "output_tokens",         
    "is_safe",
    "attempts",
    "medication_alert",
    "insulin_units_recommended",
    "insulin_timing_recommended",
    "meal_recommended",
    "exercise_recommended",
    "violations",
    "status",
    "readable_output"
]


def reset_test_csv():
    if os.path.exists(TEST_CSV_FILE):
        os.remove(TEST_CSV_FILE)
        print(f"🗑️  Deleted old {TEST_CSV_FILE}")
    with open(TEST_CSV_FILE, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=TEST_CSV_HEADERS)
        writer.writeheader()
    print(f"✅ Created {TEST_CSV_FILE} with {len(TEST_CSV_HEADERS)} columns")

def get_glucose_category(glucose):
    try:
        g = float(glucose)
        if g < 70:   return "hypoglycemia"
        if g <= 150: return "normal"
        return "hyperglycemia"
    except Exception:
        return "unknown"

def extract_field(input_str, field_name):
    for line in input_str.splitlines():
        stripped = line.strip()
        if stripped.startswith(field_name):
            parts = stripped.split("=", 1)
            if len(parts) == 2:
                return parts[1].strip()
    return "unknown"

def extract_meal_timing(label):
    for keyword in ["BEFORE", "AFTER", "AT", "PAST"]:
        if keyword in label:
            for part in label.split("|"):
                if keyword in part:
                    return part.strip()
    return "unknown"

def parse_output_fields(result):
    current_glucose = glucose_at_meal_time = carb_rule = None
    insulin_units = insulin_timing = meal_recommended = None
    exercise_recommended = medication_alert = None
    is_safe = attempts = violations = None
    status = "unknown"

    if not isinstance(result, dict):
        return {k: None for k in [
            "current_glucose","glucose_at_meal_time","carb_rule",
            "is_safe","attempts","medication_alert",
            "insulin_units_recommended","insulin_timing_recommended",
            "meal_recommended","exercise_recommended",
            "violations","status"
        ]} | {"violations": "[]", "status": "unknown"}

    status   = result.get("status", "unknown")
    attempts = result.get("attempts", None)

    if status == "safe":
        is_safe  = True
        violations = "[]"
        output   = result.get("structured_output", {})
    else:
        is_safe  = False
        violations = json.dumps(result.get("violations", []))
        output   = result.get("last_output", {})

    if isinstance(output, dict):
        current_glucose      = output.get("current_glucose")
        glucose_at_meal_time = output.get("glucose_at_meal_time")
        carb_rule            = output.get("carb_rule")
        ins                  = output.get("insulin_recommendation", {}) or {}
        insulin_units        = ins.get("units")
        insulin_timing       = ins.get("timing")
        medication_alert     = str(output.get("medication_recommendation","") or "")
        meal_recommended     = str(output.get("meal_recommendation","") or "")
        ex = output.get("exercise_recommendation", {}) or {}
        if isinstance(ex, dict):
            exercise_recommended = (
                f"status={ex.get('status')} | "
                f"intensity={ex.get('max_intensity')} | "
                f"effect={ex.get('exercise_glucose_effect')} | "
                f"duration={ex.get('duration')}"
            )
        else:
            exercise_recommended = str(ex)

    return {
        "current_glucose":            current_glucose,
        "glucose_at_meal_time":       glucose_at_meal_time,
        "carb_rule":                  carb_rule,
        "is_safe":                    is_safe,
        "attempts":                   attempts,
        "medication_alert":           medication_alert,
        "insulin_units_recommended":  insulin_units,
        "insulin_timing_recommended": insulin_timing,
        "meal_recommended":           meal_recommended,
        "exercise_recommended":       exercise_recommended,
        "violations":                 violations,
        "status":                     status
    }

def append_test_result(test_id, tc, elapsed, result, token_counter=None):
    input_str = tc["input"]
    label     = tc["label"]
    parsed    = parse_output_fields(result)

    # ── Read token counts ──────────────────────────────────────────────────────
    input_tokens  = token_counter.input_tokens  if token_counter else None
    output_tokens = token_counter.output_tokens if token_counter else None

    # ── Formatter Agent output ────────────────────────────────────────────────
    # result["readable_output"] is the plain-text patient report
    # produced by FormatterAgent after a safe run.
    # For failed/errored runs it will be None.
    if isinstance(result, dict):
        readable_output = result.get("readable_output", None)
        if readable_output is None:
            # Run was not safe — store violations summary instead
            violations_list = result.get("violations", [])
            readable_output = (
                f"[NOT SAFE] {'; '.join(violations_list)}"
                if violations_list
                else f"[{result.get('status', 'unknown').upper()}]"
            )
    else:
        readable_output = str(result)

    current_glucose = parsed["current_glucose"]

    with open(TEST_CSV_FILE, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=TEST_CSV_HEADERS)
        writer.writerow({
            "run_timestamp":              datetime.utcnow().isoformat(timespec="seconds") + "Z",
            "test_id":                    test_id,
            "label":                      label,
            "row_number":                 extract_field(input_str, "row_number"),
            "current_glucose":            current_glucose,
            "glucose_at_meal_time":       parsed["glucose_at_meal_time"],
            "glucose_category":           get_glucose_category(current_glucose),
            "carb_rule":                  parsed["carb_rule"],
            "meal_timing":                extract_meal_timing(label),
            "oral_medication":            extract_field(input_str, "oral_medication"),
            "insulin":                    extract_field(input_str, "insulin"),
            "long_acting_insulin":        extract_field(input_str, "long_acting_insulin"),
            "glp1":                       extract_field(input_str, "glp1"),
            "diet":                       extract_field(input_str, "diet"),
            "elapsed_seconds":            round(elapsed, 2),
            "input_tokens":               input_tokens,
            "output_tokens":              output_tokens,
            "is_safe":                    parsed["is_safe"],
            "attempts":                   parsed["attempts"],
            "medication_alert":           parsed["medication_alert"],
            "insulin_units_recommended":  parsed["insulin_units_recommended"],
            "insulin_timing_recommended": parsed["insulin_timing_recommended"],
            "meal_recommended":           parsed["meal_recommended"],
            "exercise_recommended":       parsed["exercise_recommended"],
            "violations":                 parsed["violations"],
            "status":                     parsed["status"],
            "readable_output":            readable_output,  
        })


In [26]:
# ─── CELL 9: TEST RUNNER ──────────────────────────────────────────────────────

async def run_all_tests(test_cases):
    reset_test_csv()
    summary = {"total": len(test_cases), "passed": 0, "failed": 0, "errored": 0}

    for i, tc in enumerate(test_cases):
        test_id = f"TC-{str(i+1).zfill(3)}"
        elapsed = 0
        print(f"\n{'='*60}")
        print(f"[{i+1}/{len(test_cases)}] {test_id}: {tc['label']}")
        print(f"{'='*60}")

        try:
            token_counter.reset()          # ← reset before each run

            start  = time.time()
            result = await run_main_with_safety(tc["input"])
            elapsed = time.time() - start

            print(f"   Tokens — input: {token_counter.input_tokens:,} | "
                  f"output: {token_counter.output_tokens:,}")

            if isinstance(result, dict) and result.get("status") == "failed":
                summary["failed"] += 1
                print(f"❌ Failed  | {elapsed:.2f}s | {result.get('violations')}")
            else:
                summary["passed"] += 1
                print(f"✅ Passed  | {elapsed:.2f}s")

            append_test_result(
                test_id,
                tc,
                elapsed,
                result,
                token_counter=token_counter    # ← pass counter
            )

        except Exception as e:
            elapsed = time.time() - start if elapsed == 0 else elapsed
            summary["errored"] += 1
            error_result = {"status": "error", "violations": [str(e)], "last_output": {}}
            append_test_result(
                test_id,
                tc,
                elapsed,
                error_result,
                token_counter=token_counter    # ← pass even on error
            )
            print(f"💥 Error   | {elapsed:.2f}s | {e}")

    print(f"\n{'='*60}")
    print(f"TEST RUN COMPLETE")
    print(f"  Total:      {summary['total']}")
    print(f"  ✅ Passed:  {summary['passed']}")
    print(f"  ❌ Failed:  {summary['failed']}")
    print(f"  💥 Errored: {summary['errored']}")
    print(f"  Results → {TEST_CSV_FILE}")
    print(f"{'='*60}")
    return summary


In [ ]:
# ─── CELL 10: RUN ─────────────────────────────────────────────────────────────

# Uncomment ONE of the options below:
MAX_RETRIES = 3
# Option A: run all test cases
summary = await run_all_tests(test_cases)

# Option B: run a single test case for quick debugging
# tc = test_cases[0]
# print(f"Running: {tc['label']}")
# result = await run_main_with_safety(tc["input"])
# print(result)

# Option C: run a specific group only
# group = [tc for tc in test_cases if tc['label'].startswith('HYPO')]
# summary = await run_all_tests(group)

✅ Created test_results.csv with 27 columns

[1/50] TC-001: NORM-BRK-01 | Normal→Normal | Monday 30min BEFORE breakfast | oral+insulin+LAI

========== ATTEMPT 1 ==========

 ### Created new session: debug_session_id

User > {"user_input": "\nlast_meal = Dinner at 7:00 PM ET previous day\ncurrent_time = Monday, 6:30 AM ET\nrow_number = 80\nweight = 75 kg\nheight = 1.65 m\ndiet = Non-Veg\nusual_meal_times:\n  breakfast = 7:00 AM ET\n  lunch = 12:30 PM ET\n  dinner = 7:00 PM ET\noral_medication = pre-meal\ninsulin = yes\nlong_acting_insulin = Yes, every night 9PM ET\nglp1 = no\n", "previous_output": null, "violations": []}
[logging_plugin] 🚀 USER MESSAGE RECEIVED
[logging_plugin]    Invocation ID: e-a4957dba-5842-4610-8e7e-a4925c48a206
[logging_plugin]    Session ID: debug_session_id
[logging_plugin]    User ID: debug_user_id
[logging_plugin]    App Name: InMemoryRunner
[logging_plugin]    Root Agent: Orchestrator_Agent
[logging_plugin]    User Content: text: '{"user_input": "\nlast_meal =

/opt/conda/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


[logging_plugin] 🧠 LLM RESPONSE
[logging_plugin]    Agent: Orchestrator_Agent
[logging_plugin]    Content: function_call: InsulinRecommenderAgent
[logging_plugin]    Token Usage - Input: 4827, Output: 34
[logging_plugin] 📢 EVENT YIELDED
[logging_plugin]    Event ID: 909bd6f4-03fc-473d-bf01-c18671ba9859
[logging_plugin]    Author: Orchestrator_Agent
[logging_plugin]    Content: function_call: InsulinRecommenderAgent
[logging_plugin]    Final Response: False
[logging_plugin]    Function Calls: ['InsulinRecommenderAgent']
[logging_plugin] 🔧 TOOL STARTING
[logging_plugin]    Tool Name: InsulinRecommenderAgent
[logging_plugin]    Agent: Orchestrator_Agent
[logging_plugin]    Function Call ID: adk-d32f1d2e-7aaa-4929-820b-72e85f13ac42
[logging_plugin]    Arguments: {'request': "The patient's glucose level at meal time is 129 mg/dL."}
[logging_plugin] 🚀 USER MESSAGE RECEIVED
[logging_plugin]    Invocation ID: e-da6a86d6-68a4-4606-8bd4-c95f9a21ae37
[logging_plugin]    Session ID: 7bc875db-2d08-

/opt/conda/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


[logging_plugin] 🧠 LLM RESPONSE
[logging_plugin]    Agent: Orchestrator_Agent
[logging_plugin]    Content: function_call: MealRecommenderAgent
[logging_plugin]    Token Usage - Input: 4605, Output: 90
[logging_plugin] 📢 EVENT YIELDED
[logging_plugin]    Event ID: ff80259c-708c-46eb-9311-4403b16836d5
[logging_plugin]    Author: Orchestrator_Agent
[logging_plugin]    Content: function_call: MealRecommenderAgent
[logging_plugin]    Final Response: False
[logging_plugin]    Function Calls: ['MealRecommenderAgent']
[logging_plugin] 🔧 TOOL STARTING
[logging_plugin]    Tool Name: MealRecommenderAgent
[logging_plugin]    Agent: Orchestrator_Agent
[logging_plugin]    Function Call ID: adk-a8678230-5fd6-4a4e-9936-b2f6e12e6099
[logging_plugin]    Arguments: {'request': 'glucose_at_meal_time: 124.2, predicted glucose trajectory: [124.7, 122.5, 128.7, 129.6], diet preference: Non-Veg, meal_type: Breakfast, glucose target range: 90–150 mg/dL'}
[logging_plugin] 🚀 USER MESSAGE RECEIVED
[logging_plugin

...

[logging_plugin] 🧠 LLM RESPONSE
[logging_plugin]    Agent: Orchestrator_Agent
[logging_plugin]    Content: function_call: MealRecommenderAgent
[logging_plugin]    Token Usage - Input: 4596, Output: 98
[logging_plugin] 📢 EVENT YIELDED
[logging_plugin]    Event ID: 5baa6be2-ab44-4e6c-879f-bc07f39d22cc
[logging_plugin]    Author: Orchestrator_Agent
[logging_plugin]    Content: function_call: MealRecommenderAgent
[logging_plugin]    Final Response: False
[logging_plugin]    Function Calls: ['MealRecommenderAgent']
[logging_plugin] 🔧 TOOL STARTING
[logging_plugin]    Tool Name: MealRecommenderAgent
[logging_plugin]    Agent: Orchestrator_Agent
[logging_plugin]    Function Call ID: adk-a52804ed-a081-42f6-b8b0-d799fcaf72c4
[logging_plugin]    Arguments: {'request': 'glucose_at_meal_time: 59.4. predicted glucose trajectory: [60.9, 81, 99.3, 102.3]. diet preference: Non-Veg. meal_type: Lunch. glucose target range: 90–150 mg/dL. carb_rule: LOW (fast-acting carbohydrates REQUIRED immediately).'}


/opt/conda/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


[logging_plugin] 🧠 LLM RESPONSE
[logging_plugin]    Agent: Orchestrator_Agent
[logging_plugin]    Content: function_call: ExerciseRecommenderAgent
[logging_plugin]    Token Usage - Input: 4820, Output: 100
[logging_plugin] 📢 EVENT YIELDED
[logging_plugin]    Event ID: d28ca247-312b-4c5a-ac5b-0266e8c99078
[logging_plugin]    Author: Orchestrator_Agent
[logging_plugin]    Content: function_call: ExerciseRecommenderAgent
[logging_plugin]    Final Response: False
[logging_plugin]    Function Calls: ['ExerciseRecommenderAgent']
[logging_plugin] 🔧 TOOL STARTING
[logging_plugin]    Tool Name: ExerciseRecommenderAgent
[logging_plugin]    Agent: Orchestrator_Agent
[logging_plugin]    Function Call ID: adk-c7dfcc9c-7332-48c9-b24c-8ce415cda333
[logging_plugin]    Arguments: {'request': 'glucose_level: 91.8. minutes_since_last_meal: 60. last_meal_carbs: None. upcoming_meal_carbs: None. predicted_glucose_trend: [97.6, 106.4, 115.9, 114.8]. min_predicted_glucose: 97.6.'}
[logging_plugin] 🚀 USER MESS

/opt/conda/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


[logging_plugin] 🧠 LLM RESPONSE
[logging_plugin]    Agent: Orchestrator_Agent
[logging_plugin]    Content: function_call: ExerciseRecommenderAgent
[logging_plugin]    Token Usage - Input: 4602, Output: 101
[logging_plugin] 📢 EVENT YIELDED
[logging_plugin]    Event ID: bdf79355-bcdb-4824-8175-0eae1d52d18b
[logging_plugin]    Author: Orchestrator_Agent
[logging_plugin]    Content: function_call: ExerciseRecommenderAgent
[logging_plugin]    Final Response: False
[logging_plugin]    Function Calls: ['ExerciseRecommenderAgent']
[logging_plugin] 🔧 TOOL STARTING
[logging_plugin]    Tool Name: ExerciseRecommenderAgent
[logging_plugin]    Agent: Orchestrator_Agent
[logging_plugin]    Function Call ID: adk-603d2550-eb0b-4025-926d-e41eba9b77cf
[logging_plugin]    Arguments: {'request': 'glucose_level: 135. minutes_since_last_meal: 59. last_meal_carbs: None. upcoming_meal_carbs: None. predicted_glucose_trend: [133.9, 134.9, 129.5, 123.2]. min_predicted_glucose: 123.2.'}
[logging_plugin] 🚀 USER MES